In [ ]:
import pandas as pd

print("teste carlos")

bases = []

# 2025 - dados zipados da ANP em arquivos semestrais CSV

for semestre in [1, 2]:

    url = f"https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsas/ca/ca-2025-0{semestre}.zip"

    df = pd.read_csv(url, sep=";", encoding="latin1", compression="zip")

    bases.append(df)


# 2026 - arquivos mensais CSV

urls_2026 = [
    
    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsan/2026/01-dados-abertos-precos-gasolina-etanol.csv",

    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsan/2026/02-cados-abertos-preco-gasolina-etanol.csv",

    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsan/2026/03-dados-abertos-precos-gasolina-etanol.csv"
]


for url in urls_2026:

    df = pd.read_csv(url, sep=";", encoding="latin1")

    bases.append(df)


# juntar todos os dataframes

dados = pd.concat(bases, ignore_index=True)


C:\Users\enzod\AppData\Local\Temp\ipykernel_8272\1273084926.py:11: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, sep=";", encoding="latin1", compression="zip")


In [ ]:
dados.head()

,ï»¿Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,NE,CE,SOBRAL,ECONOGÃS DO BRASIL DIST. DERIV. DE PET. BIOC....,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,NaN,CENTRO,62010-000,GASOLINA,01/01/2025,"6,29",NaN,R$ / litro,RAIZEN
1,NE,CE,SOBRAL,ECONOGÃS DO BRASIL DIST. DERIV. DE PET. BIOC....,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,NaN,CENTRO,62010-000,GASOLINA ADITIVADA,01/01/2025,"6,49",NaN,R$ / litro,RAIZEN
2,NE,CE,SOBRAL,ECONOGÃS DO BRASIL DIST. DERIV. DE PET. BIOC....,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,NaN,CENTRO,62010-000,DIESEL S10,01/01/2025,"6,19",NaN,R$ / litro,RAIZEN
3,NE,CE,SOBRAL,ECONOGÃS DO BRASIL DIST. DERIV. DE PET. BIOC....,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,NaN,CENTRO,62010-000,ETANOL,01/01/2025,"5,19",NaN,R$ / litro,RAIZEN
4,NE,CE,SOBRAL,V.C.EMPREENDIMENTOS LTDA,03.551.935/0002-35,AVENIDA JOSE EUCLIDES FERREIRA GOMES,30,POSTO FLASH,CORACAO DE JESUS,62043-070,GASOLINA,01/01/2025,"6,53",NaN,R$ / litro,RAIZEN


In [ ]:
dados.dtypes

ï»¿Regiao - Sigla     object
Estado - Sigla        object
Municipio             object
Revenda               object
CNPJ da Revenda       object
Nome da Rua           object
Numero Rua            object
Complemento           object
Bairro                object
Cep                   object
Produto               object
Data da Coleta        object
Valor de Venda        object
Valor de Compra      float64
Unidade de Medida     object
Bandeira              object
dtype: object

In [ ]:
dados["Valor de Venda"] = (
    dados["Valor de Venda"]
    .str.replace(",", ".")
    .astype(float)
)


# converter data para datetime

dados["Data da Coleta"] = pd.to_datetime(
    dados["Data da Coleta"],
    dayfirst=True
)

dados.dtypes

ï»¿Regiao - Sigla            object
Estado - Sigla               object
Municipio                    object
Revenda                      object
CNPJ da Revenda              object
Nome da Rua                  object
Numero Rua                   object
Complemento                  object
Bairro                       object
Cep                          object
Produto                      object
Data da Coleta       datetime64[ns]
Valor de Venda              float64
Valor de Compra             float64
Unidade de Medida            object
Bandeira                     object
dtype: object

In [ ]:
# filtrar gasolina e diesel

df = dados[dados["Produto"].isin(["GASOLINA", "Gasolina Aditivada", "ETANOL"])]


# criar coluna de mês

df["Mes"] = df["Data da Coleta"].dt.to_period("M")


# calcular média

tabela = df.groupby(["Mes", "Produto"])["Valor de Venda"].mean().unstack()


# mostrar tabela
print("--- Média do Valor de Venda ---")
print()
print(tabela)

--- Média do Valor de Venda ---

Produto    ETANOL  GASOLINA
Mes                        
2025-01  4.368597  6.183839
2025-02  4.545515  6.367177
2025-03  4.542583  6.349952
2025-04  4.506405  6.316235
2025-05  4.483594  6.283602
2025-06  4.420170  6.226324
2025-07  4.376350  6.211219
2025-08  4.365642  6.182066
2025-09  4.423241  6.191606
2025-10  4.449408  6.215174
2025-11  4.444227  6.187876
2025-12  4.530275  6.203292
2026-01  4.708790  6.325559
2026-02  4.768278  6.311746
2026-03  4.864459  6.602842


C:\Users\enzod\AppData\Local\Temp\ipykernel_8272\3306909034.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Mes"] = df["Data da Coleta"].dt.to_period("M")
